# 05 — Feature Engineering & Data Split

## Goal

This notebook exists to **test two things**:

1. **The created features** — that every column produced by `src/build_features.py` is what it claims to be. Correct grain, correct derivation from the source columns, no missing values, no impossible values, and no column that silently encodes something other than its name.
2. **The train / validation / test split** — that the split is reproducible, that it partitions the customers cleanly, and that the three parts are comparable enough for a model tuned on one to be evaluated fairly on another.

Neither is exploratory. `04_eda_first_txn.ipynb` asked *what does the data say*; this notebook asks *is what we built correct*. Every cell here should be a check with a pass/fail reading, not a chart to interpret.

**Source:** `data/processed/first_transaction_churn_clean.csv` — the cleaned line-item table, one row per line item of each customer's first invoice, written by `src/correct_data_issues.py`. Everything else in this notebook is built from it by calling `src/build_features.py`, so the features are tested as the functions that produce them rather than as a CSV taken on trust.

## What is being tested

### The features

`build_features.py` collapses 125,042 corrected line items into 5,044 customer rows across 16 columns. The checks below verify each group against the line-item source it came from, rather than trusting the aggregation:

| Group | Columns | What has to hold |
|---|---|---|
| Keys & label | `customer_id`, `churn` | one row per customer; label binary and unchanged from the source |
| Calendar parts | `year`, `month`, `day_of_month`, `weekday`, `hour`, `time_of_day` | each recomputable from `first_date`; buckets total and within domain |
| Category | `country`, `country_raw` | every named country above the frequency floor; pooled rows all `Other`; the raw column regroups back to the derived one |
| Roll-ups | `n_lines`, `n_distinct_products`, `total_quantity`, `total_spend`, `avg_unit_price` | each equal to the aggregation recomputed directly from the line items |

### The split

Whatever strategy is chosen, the split has to satisfy the same properties:

- **Complete and disjoint** — every customer lands in exactly one of train / validation / test; no customer appears twice.
- **Split on the customer** — the modelling grain is the customer, so the split unit must be `customer_id`. A row-level split would be meaningless here since there is already one row each, but the property is worth asserting so it stays true if the grain ever changes.
- **Reproducible** — the same seed produces the same partition on a re-run.
- **Proportioned as intended** — 70 / 15 / 15.
- **Comparable base rates** — the churn rate in each part is close enough that hyperparameters tuned on validation transfer to test. This is the property most at risk: `04` §6 shows churn moving between 31.2% and 72.4% month to month, so a time-ordered split does *not* satisfy it while a stratified one does.

### Open decisions this notebook depends on

Two choices are still unmade, and both change what the split cells should assert:

1. **Split strategy** — stratified random on `customer_id` (holds the base rate constant, isolates the pipeline comparison) versus time-ordered (realistic for deployment, but confounded by the drift above and needing a 90-day embargo between parts).
2. **The opening cohort** — the first 90 days hold 1,581 customers churning at 36.1% against 50.0% afterwards, most likely established customers whose earlier history predates the file. Keeping them, dropping them, or flagging them changes both the row count and the base rate the split has to preserve.

`src/split_data.py` does not exist yet.

In [6]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

RANDOM_SEED = 42

PROCESSED = Path('..') / 'data' / 'processed'
SRC = Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# The starting point for everything below: one row per line item of each
# customer's first invoice, after the corrections in correct_data_issues.py.
lines = pd.read_csv(PROCESSED / 'first_transaction_churn_clean.csv',
                    parse_dates=['invoice_date'])

print(f'Cleaned first transactions : {lines.shape[0]:,} rows x {lines.shape[1]} columns')
print(f'Customers                  : {lines["customer_id"].nunique():,}')
print(f'Invoices                   : {lines["invoice"].nunique():,}')
print(f'Date range                 : {lines["invoice_date"].min():%Y-%m-%d} -> '
      f'{lines["invoice_date"].max():%Y-%m-%d}')
print(f'Churn rate                 : {lines.groupby("customer_id")["churn"].first().mean():.2%}')
print()
print(lines.dtypes.to_string())

lines.head(10)

Cleaned first transactions : 125,042 rows x 10 columns
Customers                  : 5,044
Invoices                   : 5,044
Date range                 : 2009-12-01 -> 2011-09-09
Churn rate                 : 45.64%

invoice                  int64
stock_code              object
description             object
quantity                 int64
invoice_date    datetime64[ns]
price                  float64
customer_id              int64
country                 object
line_total             float64
churn                    int64


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,0
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,0
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,0
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,39.60,0
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,0
7,489434,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,59.50,0
8,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078,United Kingdom,59.50,0
9,489436,21755,LOVE BUILDING BLOCK WORD,18,2009-12-01 09:06:00,5.45,13078,United Kingdom,98.10,0


## Building the variables

Two modules turn the table above into modelling inputs, split by grain. The cells below call them directly rather than reading their output:

| Table | Grain | Function |
|---|---|---|
| Customer modelling table | one row per `customer_id` | `build_features(lines)` from `src/build_features.py` |
| Item history lookup | one row per `(stock_code, date)` | `build_item_history(lines)` from `src/build_item_history.py` |

Each has a matching `check_*` function in the same module that asserts its way through the properties the table has to satisfy; both are run here, so a cell that executes without raising is itself the first pass/fail reading.

The item table is a **lookup**, not a feature table — which is why it lives in its own module and is deliberately not joined to the customer grain. How to summarise a basket's worth of product history to one row per customer is a modelling decision. Joining it onto the line items is therefore the next cell's job, and the aggregation to customer grain is left open.

### What gets added to the line items, and what does not

- **Joined on `(stock_code, date)`** — the six `prior_*` history columns, one set per line item.
- **Derived per row** — `year`, `month`, `day_of_month`, `weekday`, `hour`, `time_of_day` from `invoice_date`, using the same `add_date_parts` the modelling table uses.
- **Mapped down from the customer grain** — `country_grouped`. The frequency floor counts *customers*, so recomputing it at line grain would weight each country by basket size; it is mapped from `features` instead of rebuilt here.
- **Deliberately not joined** — the customer-level roll-ups (`n_lines`, `total_quantity`, `total_spend`, `avg_unit_price`, …). Those are the *result* of aggregating these line items, and broadcasting them back onto the rows they were computed from is how double counting starts. They stay in `features`.

In [7]:
from build_features import add_date_parts, build_features, check_modelling_table
from build_item_history import build_item_history, check_item_history

# Customer grain: one row per customer, the shared starting point for both pipelines.
features = build_features(lines)
check_modelling_table(features)

# (stock_code, date) grain: how each product had traded strictly before that date.
items = build_item_history(lines)
check_item_history(items, lines)

print(f'features : {features.shape[0]:,} rows x {features.shape[1]} columns  '
      f'(checks passed)')
print(f'items    : {items.shape[0]:,} rows x {items.shape[1]} columns  '
      f'(checks passed)')

# Both were built from `lines` here; the module also writes them to disk. Read the
# item lookup back to confirm the file and the function agree, then preview it.
items_file = pd.read_csv(PROCESSED / 'item_history.csv',
                         parse_dates=['date'])
assert items_file.shape == items.shape, 'item_history.csv is a different shape'
pd.testing.assert_frame_equal(items_file, items.reset_index(drop=True))
print('\nitem_history.csv matches build_item_history(lines)')

print(f'\nDistinct stock codes     : {items["stock_code"].nunique():,}')
print(f'Rows with no history yet : {(items["prior_transactions"] == 0).sum():,} '
      f'({(items["prior_transactions"] == 0).mean()*100:.1f}%) — every product\'s '
      f'first appearance')

items_file.head(10)

features : 5,044 rows x 16 columns  (checks passed)
items    : 100,158 rows x 8 columns  (checks passed)

item_history.csv matches build_item_history(lines)

Distinct stock codes     : 4,171
Rows with no history yet : 4,171 (4.2%) — every product's first appearance


,stock_code,date,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price
0,10002,2009-12-01,0,0,NaN,NaN,NaN,NaN
1,10002,2009-12-03,1,12,1.00,12.00,0.85,0.85
2,10002,2009-12-04,4,19,2.00,9.50,0.85,0.85
3,10002,2009-12-06,8,92,3.00,12.00,0.85,0.85
4,10002,2009-12-07,9,140,2.00,30.00,0.85,0.85
5,10002,2009-12-11,10,142,1.00,12.00,0.85,0.85
6,10002,2009-12-14,11,151,1.00,10.50,0.85,0.85
7,10002,2010-01-04,13,187,1.00,12.00,0.85,0.85
8,10002,2010-01-11,14,190,1.00,10.50,0.85,0.85
9,10002,2010-01-14,15,238,1.00,12.00,0.85,0.85


In [8]:
items_file.head()

,stock_code,date,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price
0,10002,2009-12-01,0,0,NaN,NaN,NaN,NaN
1,10002,2009-12-03,1,12,1.00,12.00,0.85,0.85
2,10002,2009-12-04,4,19,2.00,9.50,0.85,0.85
3,10002,2009-12-06,8,92,3.00,12.00,0.85,0.85
4,10002,2009-12-07,9,140,2.00,30.00,0.85,0.85


## Item history applied to the first transactions

Applying `src/build_features.py` to the cleaned first-transaction data, so every line item carries the trading history its product had **before** that transaction's date.

### How the metrics move over time

The module produces one row per `(stock_code, date)`, each computed from data strictly earlier than that date. A product bought on 3 March and again on 6 March therefore gets two different rows — the first sees everything up to 2 March, the second everything up to 5 March. The history grows as the product accumulates trade:

| `stock_code` | `date` | `prior_transactions` | `prior_units` |
|---|---|---|---|
| 85123A | 2009-12-01 | 0 | 0 |
| 85123A | 2009-12-03 | 31 | 1,003 |
| 85123A | 2009-12-06 | 61 | 1,415 |

Same-day transactions share a cutoff: because history is aggregated to whole days before any accumulation, all 18 customers who bought `85123A` on 2009-12-08 read the same 79 prior transactions, and none of them sees the other 17.

### The six history metrics

**Volume — how much the product had sold**

- **`prior_transactions`** — transactions containing this product before today.
- **`prior_units`** — units of it sold before today.
- **`prior_median_daily_transactions`** — on a typical earlier trading day, how many transactions included it.
- **`prior_median_daily_units`** — on a typical earlier trading day, how many units moved.

**Price level — what the product normally cost**

- **`prior_avg_price`** — mean, across earlier trading days, of that day's average unit price.
- **`prior_median_daily_price`** — median of the same daily prices, so a one-off promotion does not drag the level.

Both price columns are built from **daily** averages rather than from raw line items, matching how the unit metrics work. A day on which forty customers bought counts once, exactly as a one-customer day does — otherwise busy days would define "the usual price". This matters because `04` §8a found 1,681 codes (40.3%) selling at more than one price.

### Comparing the current line against that history

Three ratio columns put the line's own values next to what the product normally does:

- **`qty_vs_median_daily_units`** = `quantity / prior_median_daily_units`. Above 1 means this single line moved more than the product's whole typical day.
- **`qty_share_of_prior_units`** = `quantity / prior_units`. What fraction of everything ever sold of this product is being bought right now.
- **`price_vs_prior_avg_price`** = `price / prior_avg_price`. Above 1 means this customer paid a premium over the product's usual level; below 1 means they caught it discounted.

All three are `NaN` on a product's first-ever appearance, where there is no history to divide by. That is roughly 4% of rows and is a genuine undefined, not a missing value to impute.

In [10]:
# Item history onto every line item. (stock_code, date) is the lookup's key, so a
# many-to-one merge is the whole join: several line items may share a product-day.
txn = lines.assign(date=lines['invoice_date'].dt.normalize())
txn = txn.merge(items, on=['stock_code', 'date'], how='left', validate='many_to_one')

assert len(txn) == len(lines), 'the merge changed the row count'
assert txn['prior_transactions'].notna().all(), 'a line item found no history row'

# Calendar parts, from the same function the modelling table uses. add_date_parts
# reads a column named first_date, so hand it the invoice timestamp under that name
# and take the six columns back — the temporary frame keeps txn unmutated.
DATE_PARTS = ['year', 'month', 'day_of_month', 'weekday', 'hour', 'time_of_day']
txn[DATE_PARTS] = add_date_parts(
    txn[['invoice_date']].rename(columns={'invoice_date': 'first_date'}))[DATE_PARTS]

# Grouped country mapped down from the customer grain, not recomputed at line grain.
# `country` is already here ungrouped, so the pooled version needs its own name.
txn = txn.merge(features[['customer_id', 'country']]
                .rename(columns={'country': 'country_grouped'}),
                on='customer_id', how='left', validate='many_to_one')

assert txn['country_grouped'].notna().all(), 'a line item found no country'
assert (txn.groupby('customer_id')['prior_transactions'].size()
        == features.set_index('customer_id')['n_lines']).all(), \
    'line counts no longer agree with n_lines'

added = [c for c in txn.columns if c not in lines.columns]
print(f'Enriched line items : {txn.shape[0]:,} rows x {txn.shape[1]} columns')
print(f'Added {len(added)} columns: {", ".join(added)}')
print()
print('Ready to aggregate: group by customer_id to reach the modelling grain.')

txn.head(5)

Enriched line items : 125,042 rows x 24 columns
Added 14 columns: date, prior_transactions, prior_units, prior_median_daily_transactions, prior_median_daily_units, prior_avg_price, prior_median_daily_price, year, month, day_of_month, weekday, hour, time_of_day, country_grouped

Ready to aggregate: group by customer_id to reach the modelling grain.


,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,line_total,churn,date,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price,year,month,day_of_month,weekday,hour,time_of_day,country_grouped
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,0,2009-12-01,0,0,NaN,NaN,NaN,NaN,2009,12,1,1,7,morning,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0,2009-12-01,0,0,NaN,NaN,NaN,NaN,2009,12,1,1,7,morning,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,0,2009-12-01,0,0,NaN,NaN,NaN,NaN,2009,12,1,1,7,morning,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,0,2009-12-01,0,0,NaN,NaN,NaN,NaN,2009,12,1,1,7,morning,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,0,2009-12-01,0,0,NaN,NaN,NaN,NaN,2009,12,1,1,7,morning,United Kingdom


In [11]:
# --------------------------------------------------------------------------
# Line-vs-history ratios, built on top of the joined columns
# --------------------------------------------------------------------------
# Current line vs the product's own history. Guard the denominators: prior_units is
# 0 on a first appearance, and the medians and price mean are NaN there, so the
# ratios come out NaN rather than infinite.
txn['qty_vs_median_daily_units'] = (
    txn['quantity'] / txn['prior_median_daily_units'].replace(0, np.nan))
txn['qty_share_of_prior_units'] = (
    txn['quantity'] / txn['prior_units'].replace(0, np.nan))
txn['price_vs_prior_avg_price'] = (
    txn['price'] / txn['prior_avg_price'].replace(0, np.nan))

print(f'\nLine items with history joined : {len(txn):,}')
print(f'Ratios undefined (first appearance of the product): '
      f'{txn["qty_share_of_prior_units"].isna().sum():,} '
      f'({txn["qty_share_of_prior_units"].isna().mean()*100:.1f}%)')


Line items with history joined : 125,042
Ratios undefined (first appearance of the product): 5,659 (4.5%)


In [13]:
ITEM_HISTORY_COLUMNS = [
    'date',
    'customer_id',
    'stock_code',
    'quantity',
    'price',
    'prior_transactions',
    'prior_units',
    'prior_median_daily_transactions',
    'prior_median_daily_units',
    'prior_avg_price',
    'prior_median_daily_price',
    'qty_vs_median_daily_units',
    'qty_share_of_prior_units',
    'price_vs_prior_avg_price',
]

item_history = txn[ITEM_HISTORY_COLUMNS].sort_values(['date', 'customer_id', 'stock_code'])
print(f'{len(item_history):,} rows x {item_history.shape[1]} columns')

# The opening day is every product's first appearance, so it is all NaN by
# construction and makes a poor preview. Show a mid-period transaction instead.
mid = item_history[item_history['prior_transactions'] > 0]
example_customer = mid.iloc[len(mid) // 2]['customer_id']

print(f'\nOne complete transaction — customer {example_customer}:')
item_history[item_history['customer_id'] == example_customer].head(5)

125,042 rows x 14 columns

One complete transaction — customer 16894:


,date,customer_id,stock_code,quantity,price,prior_transactions,prior_units,prior_median_daily_transactions,prior_median_daily_units,prior_avg_price,prior_median_daily_price,qty_vs_median_daily_units,qty_share_of_prior_units,price_vs_prior_avg_price
64371,2010-06-22,16894,15036,12,0.75,57,2827,1.00,24.00,0.70,0.75,0.50,0.00,1.08
64229,2010-06-22,16894,15044A,1,2.95,23,110,1.00,6.00,2.95,2.95,0.17,0.01,1.00
64228,2010-06-22,16894,15044B,1,2.95,18,124,1.00,6.00,2.93,2.95,0.17,0.01,1.01
64227,2010-06-22,16894,15044C,1,2.95,18,75,1.00,3.00,2.95,2.95,0.33,0.01,1.00
64212,2010-06-22,16894,15056BL,2,5.95,77,1196,1.00,3.00,5.85,5.95,0.67,0.00,1.02


In [ ]:

# mid['customer_id'].unique()
mid[mid['customer_id'] == 12437]
# mid